# Survival Analysis

Time-to-event modelling with censoring. Used for churn, medical outcomes,
equipment failure. Key insight: **censored observations are not missing data**
— they carry information ("survived at least this long").

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

## 1. Kaplan-Meier curves

Non-parametric survival function estimator. Handles censoring correctly.
Compare curves between groups with the log-rank test.

In [ ]:
rng = np.random.default_rng(42)
n   = 600

# Simulate customer churn data
monthly_plan = rng.binomial(1, 0.55, n).astype(bool)
base_rate    = np.where(monthly_plan, 0.08, 0.03)   # monthly churn hazard
tenure       = np.zeros(n)
event        = np.zeros(n, dtype=bool)

for i in range(n):
    for month in range(1, 37):
        if rng.random() < base_rate[i]:
            tenure[i] = month
            event[i]  = True
            break
    if not event[i]:
        tenure[i] = 36   # censored at study end

df = pd.DataFrame({
    "tenure": tenure, "event": event,
    "plan":   np.where(monthly_plan, "Monthly", "Annual"),
    "charges": rng.lognormal(5, 0.4, n),
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# KM curves by plan type
ax = axes[0]
colors_plan = {"Monthly": C[1], "Annual": C[2]}
kmf = KaplanMeierFitter()
for plan, col in colors_plan.items():
    mask = df["plan"] == plan
    kmf.fit(df.loc[mask, "tenure"], df.loc[mask, "event"], label=plan)
    kmf.plot_survival_function(ax=ax, color=col, ci_show=True, ci_alpha=0.15)

# Log-rank test
m_mask = df["plan"] == "Monthly"
lr = logrank_test(df.loc[m_mask, "tenure"],  df.loc[~m_mask, "tenure"],
                  df.loc[m_mask, "event"],   df.loc[~m_mask, "event"])
ax.set(xlabel="Months", ylabel="Survival probability",
       title=f"Kaplan-Meier curves by plan type
Log-rank p={lr.p_value:.4f}")
ax.legend(title="Plan type")

# At-risk table proxy (survival at key months)
ax = axes[1]
months = [6, 12, 18, 24, 30, 36]
for plan, col in colors_plan.items():
    mask = df["plan"] == plan
    kmf.fit(df.loc[mask, "tenure"], df.loc[mask, "event"])
    surv_at_months = [kmf.predict(m) for m in months]
    ax.plot(months, surv_at_months, "o-", color=col, lw=2, label=plan)
    for m, s in zip(months, surv_at_months):
        ax.annotate(f"{s:.0%}", xy=(m, s), xytext=(0, 8),
                    textcoords="offset points", ha="center", fontsize=8, color=col)

ax.set(xlabel="Month", ylabel="Survival probability",
       title="Survival probability at key months
(Annual plan retains much better)")
ax.legend(title="Plan type")
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()
print(f"Log-rank test p-value: {lr.p_value:.6f}")
print(f"Monthly 12-month survival:  {kmf.predict(12):.1%}")

## 2. Cox Proportional Hazards — covariate effects

Semi-parametric model: estimates **hazard ratios** (HR) for each covariate.
HR > 1 = increases hazard (faster churn); HR < 1 = protective.

In [ ]:
from lifelines import CoxPHFitter

df_cox = df.copy()
df_cox["monthly_plan"] = (df_cox["plan"] == "Monthly").astype(int)
df_cox = df_cox.drop(columns=["plan"])

cph = CoxPHFitter()
cph.fit(df_cox, duration_col="tenure", event_col="event")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Coefficient forest plot
ax = axes[0]
summary = cph.summary
coefs   = summary["coef"]
se      = summary["se(coef)"]
hr      = np.exp(coefs)
hr_lo   = np.exp(coefs - 1.96*se)
hr_hi   = np.exp(coefs + 1.96*se)

y_pos = range(len(coefs))
ax.errorbar(hr, y_pos, xerr=[hr-hr_lo, hr_hi-hr], fmt="o",
            color=C[0], ms=8, capsize=5, lw=2)
ax.axvline(1.0, color="grey", lw=1.5, linestyle="--", label="HR=1 (no effect)")
ax.set(yticks=list(y_pos), yticklabels=coefs.index,
       xlabel="Hazard Ratio (95% CI)",
       title="Cox PH: Hazard Ratios
(HR>1 = faster churn, HR<1 = slower)")
ax.legend()
for i, (h, lo, hi, name) in enumerate(zip(hr, hr_lo, hr_hi, coefs.index)):
    color = C[1] if h > 1 else C[2]
    ax.scatter(h, i, color=color, s=80, zorder=5)

# Predicted survival for representative customers
ax = axes[1]
profiles = pd.DataFrame({
    "monthly_plan": [1, 0, 1, 0],
    "charges":      [df["charges"].quantile(0.25)] * 2 + [df["charges"].quantile(0.75)] * 2,
})
labels = ["Monthly, low charges", "Annual, low charges",
          "Monthly, high charges", "Annual, high charges"]
colors_prof = [C[1], C[2], C[3], C[0]]
for profile, label, col in zip(profiles.itertuples(index=False), labels, colors_prof):
    row = pd.DataFrame([profile._asdict()])
    sf  = cph.predict_survival_function(row, times=range(1, 37))
    ax.plot(sf.index, sf.values.flatten(), color=col, lw=2, label=label)

ax.set(xlabel="Months", ylabel="Survival probability",
       title="Predicted survival by customer profile
(Cox PH)")
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()
print("
Hazard ratios (exponentiated coefficients):")
print(np.exp(cph.params_).round(3))